In [1]:
from pathlib import Path
from collections import Counter

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
    CSVLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS

C:\Users\essam\AppData\Local\Temp\ipykernel_2492\3345350297.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [3]:
llm_model = "llama3.2:3b"
empide = "nomic-embed-text"


In [5]:
llm = ChatOllama(model = llm_model,temperature=0)
embedding_model = OllamaEmbeddings(model=empide)

In [ ]:
data_folder = Path(r"E:\data 2")

In [13]:
def load_all_data(data_folder):
    all_documents = []
    data_path = Path(data_folder)
    for x in data_path.rglob("*"):
        if not x.is_file():
            continue
        extenstion = x.suffix.lower()
        course_name = x.parent.name
        try:
            if extenstion ==".txt":
                loader = TextLoader(str(x),encoding="utf-8")
            elif extenstion== ".pdf":
                loader = PyPDFLoader(str(x))
            elif extenstion == ".docx":
                loader = Docx2txtLoader(str(x))
            elif extenstion == ".csv":
                loader = CSVLoader(str(x),encoding="utf-8")
            else:
                print(f"Skipped : {x.name}")
                continue
            loaded_dec = loader.load()
            for c in loaded_dec:
                c.metadata["course"]=course_name
                c.metadata["filename"]=x.name
                c.metadata["file_type"]=extenstion
            all_documents.extend(loaded_dec)
            print(
                f"Loaded: {x.name} | "
                f"Course: {course_name} | "
                f"Documents: {len(loaded_dec)}"
            )
        except Exception as error:
            print(f"Error loading {x.name}: {error}")

    return all_documents





In [14]:
documents = load_all_data(data_folder)
print("\nTotal Documents ",len(documents))

Loaded: ml_basics.txt | Course: machine_learning | Documents: 1
Loaded: supervised_learning.pdf | Course: machine_learning | Documents: 5
Loaded: python_basics.docx | Course: python | Documents: 1
Loaded: statistics_course_material.csv | Course: statistics | Documents: 41

Total Documents  48


In [15]:
course_distribution = Counter(
    document.metadata.get("course")
    for document in documents
)

print("Documents by course:\n")

for course, count in course_distribution.items():
    print(f"{course}: {count}")

Documents by course:

machine_learning: 6
python: 1
statistics: 41


In [16]:
sample_document = documents[0]

print("Content:\n")
print(sample_document.page_content[:1000])

print("\nMetadata:")
print(sample_document.metadata)

Content:

Machine Learning Basics

Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed.

Supervised learning uses labeled data. The model learns from input-output examples. Common supervised learning tasks include classification and regression.

Unsupervised learning uses unlabeled data. It searches for hidden structures and patterns. Clustering is a common unsupervised learning task.

Overfitting happens when a model learns the training data too closely and performs poorly on unseen data. It can be reduced using regularization, dropout, data augmentation, early stopping, and additional training data.

The dataset is commonly divided into training, validation, and testing sets. The training set is used to train the model, the validation set is used to tune it, and the test set measures final performance.

Metadata:
{'source': 'E:\\data 2\\machine_learning\\ml_basics.txt', 'course': 'machine_learn

In [17]:
text_spliter = RecursiveCharacterTextSplitter(chunk_size = 800,chunk_overlap =120,length_function = len ,separators=["\n\n" , "\n", ". "," ",""])
chunks = text_spliter.split_documents(documents)
print("Documents before chunking:", len(documents))
print("Chunks after chunking:", len(chunks))

Documents before chunking: 48
Chunks after chunking: 69


In [18]:
print("First chunk content:\n")
print(chunks[0].page_content)

print("\nFirst chunk metadata:")
print(chunks[0].metadata)

print("\nChunk length:")
print(len(chunks[0].page_content))

First chunk content:

Machine Learning Basics

Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed.

Supervised learning uses labeled data. The model learns from input-output examples. Common supervised learning tasks include classification and regression.

Unsupervised learning uses unlabeled data. It searches for hidden structures and patterns. Clustering is a common unsupervised learning task.

Overfitting happens when a model learns the training data too closely and performs poorly on unseen data. It can be reduced using regularization, dropout, data augmentation, early stopping, and additional training data.

First chunk metadata:
{'source': 'E:\\data 2\\machine_learning\\ml_basics.txt', 'course': 'machine_learning', 'filename': 'ml_basics.txt', 'file_type': '.txt'}

Chunk length:
689


In [19]:
chunk_distribution = Counter(chunk.metadata.get("course") for  chunk in chunks)

In [20]:
chunk_distribution

Counter({'statistics': 41, 'python': 17, 'machine_learning': 11})

In [23]:
vector_store = FAISS.from_documents(documents=chunks,embedding=embedding_model)

In [24]:
print("Stored vectors:", vector_store.index.ntotal)

Stored vectors: 69


In [25]:
VECTOR_DB_PATH = "faiss_course_index"

vector_store.save_local(VECTOR_DB_PATH)

print("Vector database saved to:", VECTOR_DB_PATH)

Vector database saved to: faiss_course_index


In [29]:
test_question = "What is the difference between a list and a tuple?"

search_results = vector_store.similarity_search(query=test_question,k=3)
print(f"Question : {test_question}" )
print("Retrieved chunks :",len(search_results))
for x , r in enumerate(search_results):
    print(f"\nChunk {x+1}")
    print(r.page_content[:500])
    print("course:",r.metadata.get("course"))
    print("course:",r.metadata.get("filename"))
    print("course:",r.metadata.get("page","N/A"))

Question : What is the difference between a list and a tuple?
Retrieved chunks : 3

Chunk 1
4. Collections: Lists, Tuples, Sets, and Dictionaries

4.1 Lists

A list is an ordered and mutable collection. Ordered means items keep a position; mutable means items can be added, removed, or replaced after creation. List indexing begins at zero, and negative indexes count backward from the end.

topics = ["variables", "loops", "functions"]
topics.append("exceptions")
topics[0] = "data types"
print(topics[0])
print(topics[-1])

4.2 Tuples

A tuple is an ordered collection that is normally trea
course: python
course: python_basics.docx
course: N/A

Chunk 2
score_text = "95"
score = int(score_text)
print(type(score))       # <class 'int'>

3. Operators and Expressions

Arithmetic operators include addition (+), subtraction (-), multiplication (*), division (/), floor division (//), remainder (%), and exponentiation (**). Comparison operators return Boolean values. Logical operators combine condi

In [30]:
def retrieve_chunks(question,course = "all",k=4):
    if course.lower()=="all":
        results = vector_store.similarity_search(query= question,k=k)
    else:
        results = vector_store.similarity_search(query=question,k=k,filter={"course":course})
    return results

In [31]:
retrieve_chunks(
    question="What is a function?",
    course="python"
)

[Document(id='a7ccbea9-d279-4036-bc5a-f5d0f118bc12', metadata={'source': 'E:\\data 2\\python\\python_basics.docx', 'course': 'python', 'filename': 'python_basics.docx', 'file_type': '.docx'}, page_content='zip(): Iterates over two or more iterables in parallel.\n\n7. Functions\n\nA function is a reusable block of code that performs a focused task. Functions reduce repetition, improve testing, and organize programs into meaningful units. A function is defined with def and executed when it is called.\n\ndef calculate_average(values):\n    if not values:\n        return 0\n    return sum(values) / len(values)\n\nresult = calculate_average([80, 90, 100])\nprint(result)\n\nParameter: A name written in a function definition that receives an input value.\n\nArgument: The actual value supplied when calling a function.\n\nReturn value: The result sent back to the caller by the return statement.\n\nDefault argument: A parameter value used when the caller does not supply that argument.'),
 Docume

In [32]:
filtered_results = retrieve_chunks(
    question="What is a function?",
    course="python",
    k=3
)

for result in filtered_results:
    print(result.metadata.get("course"))
    print(result.metadata.get("filename"))
    print(result.page_content[:300])
    print("-" * 50)

python
python_basics.docx
zip(): Iterates over two or more iterables in parallel.

7. Functions

A function is a reusable block of code that performs a focused task. Functions reduce repetition, improve testing, and organize programs into meaningful units. A function is defined with def and executed when it is called.

def c
--------------------------------------------------
python
python_basics.docx
Method: A function defined inside a class and called through an object or class.

13. Testing and Code Quality

Reliable programs use meaningful names, small focused functions, consistent formatting, and explicit tests. Assertions can verify assumptions during development. Automated unit tests check
--------------------------------------------------
python
python_basics.docx
Default argument: A parameter value used when the caller does not supply that argument.

Variables created inside a function normally have local scope. They can be accessed during that function call but are not automat

In [34]:
def ask_rag(question,course = "all",k=4):
    if not question.strip():
        return "Please enter a question.",[]
    result = retrieve_chunks(question,course,k)
    if not result:
        return "the answer not found in the corse materials.",[]
    context_parts = []
    for i , d in enumerate(result,start=1):
        filename = d.metadata.get("filename","Unknown source")
        page = d.metadata.get("page")
        source_label = filename
        if page is not None:
            source_label+= f"--{page+1}"
        context_part = f""" Source {i} : {source_label}
Course {d.metadata.get("course","Unknown")}

{documents.page_content}"""
        context = "\n\n".join(context_parts)
    prompt = f"""
You are a reliable course question-answering assistant.

Use only the provided course context to answer the question.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts or sources.
3. Answer clearly and concisely.
4. If the context does not contain enough information, respond exactly with:
"The answer was not found in the course materials."

Course context:
{context}

Student question:
{question}

Answer:
"""
    response= llm.invoke(prompt)
    sources = []
    for d in result:
        filename = d.metadata.get("filename","Unknown")
        page = d.metadata.get("page")

        if page is not None:
            source = f"{filename} - Page {page + 1}"
        else:
            source = filename

        if source not in sources:
            sources.append(source)

    return response.content, sources


In [36]:
def ask_rag(question, course="all", k=4):
    if not question.strip():
        return "Please enter a question.", []

    results = retrieve_chunks(
        question=question,
        course=course,
        k=k
    )

    if not results:
        return (
            "The answer was not found in the course materials.",
            []
        )

    context_parts = []

    for i, d in enumerate(results, start=1):
        filename = d.metadata.get(
            "filename",
            "Unknown source"
        )

        page = d.metadata.get("page")

        source_label = filename

        if page is not None:
            source_label += f" - Page {page + 1}"

        context_part = f"""
Source {i}: {source_label}
Course: {d.metadata.get("course", "Unknown")}

{d.page_content}
"""

        context_parts.append(context_part)

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a reliable course question-answering assistant.

Use only the provided course context to answer the question.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts or sources.
3. Answer clearly and concisely.
4. If the context does not contain enough information, respond exactly with:
"The answer was not found in the course materials."

Course context:
{context}

Student question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    sources = []

    for d in results:
        filename = d.metadata.get(
            "filename",
            "Unknown source"
        )

        page = d.metadata.get("page")

        if page is not None:
            source = f"{filename} - Page {page + 1}"
        else:
            source = filename

        if source not in sources:
            sources.append(source)

    return response.content, sources

In [39]:
answer, sources = ask_rag(
    question="Who won the football World Cup?",
    course="all",
)

print("Answer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Answer:
The answer was not found in the course materials.

Sources:
- statistics_course_material.csv
- supervised_learning.pdf - Page 3


In [40]:
import gradio as gr

In [41]:
def gradio_answer(question, course):
    answer, sources = ask_rag(
        question=question,
        course=course,
        k=4
    )

    if sources:
        sources_text = "\n".join(
            f"- {source}"
            for source in sources
        )
    else:
        sources_text = "No sources available."

    return answer, sources_text

In [42]:
with gr.Blocks(
    title="Course Q&A RAG Assistant"
) as demo:

    gr.Markdown(
        """
        # 🎓 Course Q&A RAG Assistant

        Ask questions from **Python**, **Machine Learning**, and
        **Statistics** course materials.

        The assistant answers only from the retrieved documents
        and displays the sources used.
        """
    )

    course_input = gr.Dropdown(
        choices=[
            "all",
            "python",
            "machine_learning",
            "statistics"
        ],
        value="all",
        label="Select Course"
    )

    question_input = gr.Textbox(
        label="Student Question",
        placeholder="Write your question here...",
        lines=3
    )

    ask_button = gr.Button(
        "Ask Assistant",
        variant="primary"
    )

    answer_output = gr.Textbox(
        label="Answer",
        lines=8,
        interactive=False
    )

    sources_output = gr.Textbox(
        label="Sources",
        lines=4,
        interactive=False
    )

    gr.Examples(
        examples=[
            [
                "What is the difference between a list and a tuple?",
                "python"
            ],
            [
                "How can overfitting be reduced?",
                "machine_learning"
            ],
            [
                "What is the difference between mean and median?",
                "statistics"
            ]
        ],
        inputs=[
            question_input,
            course_input
        ]
    )

    ask_button.click(
        fn=gradio_answer,
        inputs=[
            question_input,
            course_input
        ],
        outputs=[
            answer_output,
            sources_output
        ]
    )

    question_input.submit(
        fn=gradio_answer,
        inputs=[
            question_input,
            course_input
        ],
        outputs=[
            answer_output,
            sources_output
        ]
    )

In [43]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
